<a href="https://colab.research.google.com/github/Clasico-Insights/historico-clasicos/blob/main/01_base_de_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q statsbombpy mplsoccer

from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/Historico_Clasicos'   # cambia el nombre si prefieres otro
for sub in ['datos/statsbomb/eventos', 'datos/api_football', 'graficos']:
    os.makedirs(f'{BASE}/{sub}', exist_ok=True)

print(sorted(os.listdir(BASE)))
print(sorted(os.listdir(f'{BASE}/datos')))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['datos', 'graficos', 'inventario_clasicos_statsbomb.csv']
['api_football', 'statsbomb']


In [3]:
import os, warnings
import pandas as pd
from statsbombpy import sb
warnings.filterwarnings('ignore')   # quita el aviso "credentials were not supplied" (es normal con Open Data)

BASE = '/content/drive/MyDrive/Historico_Clasicos'

# Todas las temporadas de Liga y Copa del Rey que hay en StatsBomb Open Data
comps = sb.competitions()
temporadas = comps[comps['competition_name'].isin(['La Liga', 'Copa del Rey'])]
partidos = pd.concat(
    [sb.matches(competition_id=c.competition_id, season_id=c.season_id) for c in temporadas.itertuples()],
    ignore_index=True)

# Nos quedamos con los Barça-Real Madrid
equipos = ['Barcelona', 'Real Madrid']
clasicos = (partidos[partidos['home_team'].isin(equipos) & partidos['away_team'].isin(equipos)]
            .sort_values('match_date', ascending=False)
            [['match_date', 'competition', 'season', 'home_team', 'away_team',
              'home_score', 'away_score', 'match_id']]
            .reset_index(drop=True))

print(len(temporadas), 'temporadas revisadas ->', len(clasicos), 'Clásicos')

# Comparar con el inventario que subiste
inv = pd.read_csv(f'{BASE}/inventario_clasicos_statsbomb.csv')
print('Mismos match_id que el CSV:', set(clasicos['match_id']) == set(inv['match_id']))
clasicos.head()

21 temporadas revisadas -> 31 Clásicos
Mismos match_id que el CSV: True


,match_date,competition,season,home_team,away_team,home_score,away_score,match_id
0,2021-04-10,Spain - La Liga,2020/2021,Real Madrid,Barcelona,2,1,3773497
1,2020-10-24,Spain - La Liga,2020/2021,Barcelona,Real Madrid,1,3,3773585
2,2020-03-01,Spain - La Liga,2019/2020,Real Madrid,Barcelona,2,0,303470
3,2019-12-18,Spain - La Liga,2019/2020,Barcelona,Real Madrid,0,0,303596
4,2019-03-02,Spain - La Liga,2018/2019,Real Madrid,Barcelona,0,1,16196


In [4]:
CARPETA = f'{BASE}/datos/statsbomb/eventos'
os.makedirs(CARPETA, exist_ok=True)

todos = []
for p in inv.itertuples():
    ev = sb.events(match_id=p.match_id)
    ev.to_parquet(f'{CARPETA}/{p.match_id}.parquet', index=False)  # un fichero por partido
    todos.append(ev)
    print(p.fecha, p.local, p.goles_local, '-', p.goles_visitante, p.visitante, '->', len(ev), 'eventos')

eventos = pd.concat(todos, ignore_index=True)
eventos.to_parquet(f'{BASE}/datos/statsbomb/eventos_clasicos.parquet', index=False)  # los 31 juntos

print('\nPartidos:', eventos['match_id'].nunique())
print('Eventos totales:', len(eventos))
print('Ficheros en la carpeta:', len(os.listdir(CARPETA)))

2021-04-10 Real Madrid 2 - 1 Barcelona -> 3991 eventos
2020-10-24 Barcelona 1 - 3 Real Madrid -> 4213 eventos
2020-03-01 Real Madrid 2 - 0 Barcelona -> 4227 eventos
2019-12-18 Barcelona 0 - 0 Real Madrid -> 4158 eventos
2019-03-02 Real Madrid 0 - 1 Barcelona -> 3978 eventos
2018-05-06 Barcelona 2 - 2 Real Madrid -> 3831 eventos
2017-12-23 Real Madrid 0 - 3 Barcelona -> 4101 eventos
2017-04-23 Real Madrid 2 - 3 Barcelona -> 3667 eventos
2016-12-03 Barcelona 1 - 1 Real Madrid -> 3979 eventos
2016-04-02 Barcelona 1 - 2 Real Madrid -> 3700 eventos
2015-11-21 Real Madrid 0 - 4 Barcelona -> 3860 eventos
2015-03-22 Barcelona 2 - 1 Real Madrid -> 3823 eventos
2014-10-25 Real Madrid 3 - 1 Barcelona -> 3839 eventos
2014-03-23 Real Madrid 3 - 4 Barcelona -> 3905 eventos
2013-10-26 Barcelona 2 - 1 Real Madrid -> 3693 eventos
2013-03-02 Real Madrid 2 - 1 Barcelona -> 3867 eventos
2012-10-07 Barcelona 2 - 2 Real Madrid -> 3675 eventos
2012-04-21 Barcelona 1 - 2 Real Madrid -> 3616 eventos
2011-12-10

In [5]:
# Un gol es un tiro con resultado "Goal" o un evento "Own Goal For" (gol en propia, se apunta al equipo que se beneficia)
goles = eventos[((eventos['type'] == 'Shot') & (eventos['shot_outcome'] == 'Goal'))
                | (eventos['type'] == 'Own Goal For')]

def contar(match_id, equipo):
    return int(((goles['match_id'] == match_id) & (goles['team'] == equipo)).sum())

inv['gl_eventos'] = [contar(m, e) for m, e in zip(inv['match_id'], inv['local'])]
inv['gv_eventos'] = [contar(m, e) for m, e in zip(inv['match_id'], inv['visitante'])]
inv['cuadra'] = (inv['gl_eventos'] == inv['goles_local']) & (inv['gv_eventos'] == inv['goles_visitante'])

print('Partidos que cuadran:', inv['cuadra'].sum(), 'de', len(inv))
print('Goles contados en los eventos:', len(goles))
print('Goles según los resultados:', (inv['goles_local'] + inv['goles_visitante']).sum())
inv[~inv['cuadra']]   # partidos que NO cuadran (debería salir vacío)

Partidos que cuadran: 31 de 31
Goles contados en los eventos: 106
Goles según los resultados: 106


,fecha,competicion,temporada,local,visitante,goles_local,goles_visitante,match_id,eventos,fuente,gl_eventos,gv_eventos,cuadra


In [6]:
import requests, json
from google.colab import userdata

API_KEY = userdata.get('API_FOOTBALL_KEY').strip()
URL = 'https://v3.football.api-sports.io'
H = {'x-apisports-key': API_KEY}

# 1) Estado de la cuenta
s = requests.get(f'{URL}/status', headers=H).json()['response']
print('Plan:', s['subscription']['plan'], '| consultas hoy:', s['requests']['current'], 'de', s['requests']['limit_day'])

# 2) Barça (529) - Real Madrid (541), temporada a temporada
CARPETA_AF = f'{BASE}/datos/api_football'
filas = []
for temporada in [2022, 2023, 2024]:
    d = requests.get(f'{URL}/fixtures/headtohead', headers=H,
                     params={'h2h': '529-541', 'season': temporada}).json()
    with open(f'{CARPETA_AF}/h2h_{temporada}.json', 'w') as f:   # guardamos la respuesta tal cual
        json.dump(d, f)
    if d['errors']:
        print(temporada, '-> ERROR:', d['errors'])
        continue
    for x in d['response']:
        filas.append({'fecha': x['fixture']['date'][:10], 'competicion': x['league']['name'],
                      'ronda': x['league']['round'], 'local': x['teams']['home']['name'],
                      'visitante': x['teams']['away']['name'], 'goles_local': x['goals']['home'],
                      'goles_visitante': x['goals']['away'], 'fixture_id': x['fixture']['id']})

inv_af = pd.DataFrame(filas).sort_values('fecha', ascending=False).reset_index(drop=True)
print(len(inv_af), 'partidos encontrados')
inv_af

Plan: Free | consultas hoy: 15 de 100
16 partidos encontrados


,fecha,competicion,ronda,local,visitante,goles_local,goles_visitante,fixture_id
0,2025-05-11,La Liga,Regular Season - 35,Barcelona,Real Madrid,4,3,1208801
1,2025-04-26,Copa del Rey,Final,Barcelona,Real Madrid,3,2,1367758
2,2025-01-12,Super Cup,Final,Real Madrid,Barcelona,2,5,1334517
3,2024-10-26,La Liga,Regular Season - 11,Real Madrid,Barcelona,0,4,1208587
4,2024-08-03,Friendlies Clubs,Club Friendlies 5,Barcelona,Real Madrid,2,1,1277934
5,2024-04-21,La Liga,Regular Season - 32,Real Madrid,Barcelona,3,2,1038267
6,2024-01-14,Super Cup,Final,Real Madrid,Barcelona,4,1,1159314
7,2023-10-28,La Liga,Regular Season - 11,Barcelona,Real Madrid,1,2,1038054
8,2023-07-29,Friendlies Clubs,Club Friendlies 1,Barcelona,Real Madrid,3,0,1117081
9,2023-07-29,Friendlies Clubs,Club Friendlies 5,Real Madrid,Barcelona,0,3,1097382


In [7]:
oficiales = inv_af[inv_af['competicion'] != 'Friendlies Clubs'].reset_index(drop=True)
oficiales['fuente'] = 'API-Football'

print(len(oficiales), 'Clásicos oficiales')
print(oficiales['competicion'].value_counts())

oficiales.to_csv(f'{CARPETA_AF}/inventario_clasicos_apifootball.csv', index=False)
print(sorted(os.listdir(CARPETA_AF)))

12 Clásicos oficiales
competicion
La Liga         6
Copa del Rey    3
Super Cup       3
Name: count, dtype: int64
['estadisticas', 'estadisticas_clasicos_apifootball.csv', 'h2h_2022.json', 'h2h_2023.json', 'h2h_2024.json', 'inventario_clasicos_apifootball.csv']


In [8]:
import time
CARPETA_STATS = f'{CARPETA_AF}/estadisticas'
os.makedirs(CARPETA_STATS, exist_ok=True)

filas = []
for p in oficiales.itertuples():
    ruta = f'{CARPETA_STATS}/{p.fixture_id}.json'
    if os.path.exists(ruta):                      # ya descargado: no gastamos consulta
        with open(ruta) as f:
            d = json.load(f)
    else:
        d = requests.get(f'{URL}/fixtures/statistics', headers=H,
                         params={'fixture': p.fixture_id}).json()
        if d['errors']:
            print(p.fixture_id, '-> ERROR:', d['errors'])
            continue
        with open(ruta, 'w') as f:                # guardamos la respuesta tal cual
            json.dump(d, f)
        time.sleep(7)                             # pausa por el límite de consultas por minuto
    for eq in d['response']:
        fila = {'fixture_id': p.fixture_id, 'fecha': p.fecha,
                'competicion': p.competicion, 'equipo': eq['team']['name']}
        for s in eq['statistics']:
            fila[s['type']] = s['value']
        filas.append(fila)
    print(p.fecha, p.local, p.goles_local, '-', p.goles_visitante, p.visitante, '->', len(d['response']), 'equipos')

stats_af = pd.DataFrame(filas)
stats_af.to_csv(f'{CARPETA_AF}/estadisticas_clasicos_apifootball.csv', index=False)
print('\nFilas:', len(stats_af), '| Columnas:', stats_af.shape[1])
print(list(stats_af.columns))


2025-05-11 Barcelona 4 - 3 Real Madrid -> 2 equipos
2025-04-26 Barcelona 3 - 2 Real Madrid -> 2 equipos
2025-01-12 Real Madrid 2 - 5 Barcelona -> 2 equipos
2024-10-26 Real Madrid 0 - 4 Barcelona -> 2 equipos
2024-04-21 Real Madrid 3 - 2 Barcelona -> 2 equipos
2024-01-14 Real Madrid 4 - 1 Barcelona -> 2 equipos
2023-10-28 Barcelona 1 - 2 Real Madrid -> 2 equipos
2023-04-05 Barcelona 0 - 4 Real Madrid -> 2 equipos
2023-03-19 Barcelona 2 - 1 Real Madrid -> 2 equipos
2023-03-02 Real Madrid 0 - 1 Barcelona -> 2 equipos
2023-01-15 Real Madrid 1 - 3 Barcelona -> 2 equipos
2022-10-16 Real Madrid 3 - 1 Barcelona -> 2 equipos

Filas: 24 | Columnas: 22
['fixture_id', 'fecha', 'competicion', 'equipo', 'Shots on Goal', 'Shots off Goal', 'Total Shots', 'Blocked Shots', 'Shots insidebox', 'Shots outsidebox', 'Fouls', 'Corner Kicks', 'Offsides', 'Ball Possession', 'Yellow Cards', 'Red Cards', 'Goalkeeper Saves', 'Total passes', 'Passes accurate', 'Passes %', 'expected_goals', 'goals_prevented']
